In [ ]:
from enum import Enum
from pydantic import BaseModel,model_validator
from dataclasses import dataclass
from abc import ABC, abstractmethod
from typing import Callable


#Immutable
# forces pure transformation of the data
# nums already immutable
class Suit(str, Enum):
    HEARTS = "♥"
    SPADES = "♠"
    CLUBS = "♣"
    DIAMONDS = "♦"

#Immutable so that the state \times axis won't be an issue when we check rank = Rank() some other time
class Rank(str, Enum):
    A = "A"
    K = "K"
    Q = "Q"
    J = "J"
    TEN = "10"
    NINE = "9"
    EIGHT = "8"
    SEVEN = "7"
    SIX = "6"
    FIVE = "5"
    FOUR = "4"
    THREE = "3"
    TWO = "2"

#Immutable pydantic offers this
class Card(BaseModel, frozen=True):
    suit: Suit
    rank: Rank


STANDARD_DECK = tuple(
    Card(rank=r, suit=s)
    for s in Suit
    for r in Rank
)

POSITIONS = tuple(range(len(STANDARD_DECK)))

#Immutable


class DeckInterface(ABC):

    @abstractmethod
    def card_at(self, position: int) -> Card:
        pass
    
    @abstractmethod
    def position_of(self, card: Card) -> int:
        pass

    # change the permutation of the 
    @abstractmethod
    def shuffle(self, permutation):
        pass
    

class Deck(BaseModel):
    #implicit total order lattice on tuple.
    cards: tuple[Card, ...]

    model_config = {
        "frozen": True
    }

    @classmethod
    def standard(cls):
        return cls(cards=STANDARD_DECK)


    # d: P -> C, deck morphism taking position to card
    def card_at(self, position: int):
        return self.cards[position]

    # d^{-1}
    def position_of(self, card: Card):
        return self.cards.index(card)

    def compute_invariant(self):
        return 
    def __iter__(self):
        return enumerate(self.cards)

    @model_validator(mode="after")
    def validate(self):
        if len(self.cards) != 52:
            raise ValueError()

        if len(set(self.cards)) != 52:
            raise ValueError()

        return self


class ShuffleAction:
    def apply(self, shuffle, deck):
        new_cards = [None] * 52

        for position, card in deck:
            new_position = shuffle(position)
            new_cards[new_position] = card

        return Deck(tuple(new_cards))

class Shuffle(ABC):
    @abstractmethod
    def __call__(self, *args, **kwargs):
        pass

# monoid category which contains two morphisms, out shuffle and inshuffle which are pure endomorphism on P -> P
class OutShuffle(Shuffle):
    def __call__(self, i: int) -> int:
        if i < 26:
            return 2*i
        else:
            return 2*(i-26)+1

class InShuffle(Shuffle):
    def __call__(i:int)->int:
        if i < 26:
            return 2*i+1
        else:
            return 2*(i-26)

class IdShuffle(Shuffle):
    def __call__(i:int) -> int:
        return lambda x:x

MONOID_MORPHISMS = {InShuffle, OutShuffle, IdShuffle}

# The monoid composition law. #equivalent to playing/ fold
class CompositeShuffle:
    def __init__(self, *operations):
        self.operations = operations

    def __call__(self, position):
        result = position

        for op in reversed(self.operations):
            result = op(result)

        return result


# Do not let callers create invalid decks.
class DeckFactory:
    @staticmethod
    def standard():
        return Deck(tuple(STANDARD_DECK))

    @staticmethod
    def from_cards(cards):
        validate(cards)
        return Deck(tuple(cards))



generators = {
    "in": InShuffle,
    "out": OutShuffle
}

# an element of Object of free Monoid of commands. 
class ShuffleCommand:

    def __init__(self, shuffle: Shuffle):
        self.shuffle = shuffle


    def execute(self, deck: Deck) -> Deck:
        return deck.apply(self.shuffle)


# Invariant Morphism which I found to be sign, aka the induction predicate which always holds is that inshuffle out outshuffle will never reach Alternating group A_n

def signature(perm):
    visited=set()
    cycles=[]

    for i in range(52):
        if i not in visited:
            cycle=[]
            j=i

            while j not in visited:
                visited.add(j)
                cycle.append(j)
                j=perm(j)

            cycles.append(len(cycle))

    return tuple(sorted(cycles))

def sign(perm):
    cycles = signature(perm)
    return (-1) ** (52 - cycles)




In [ ]:
import random


class Simulator:


    def __init__(
        self,
        generators,
        action
    ):

        self.generators = generators
        self.action = action



    def run(
        self,
        deck,
        steps
    ):

        history = []

        current = deck


        for _ in range(steps):

            shuffle = random.choice(
                self.generators
            )

            history.append(shuffle)

            current = self.action.apply(
                current,
                shuffle
            )


        return current, history

## Attempt 1 for shuffle problem invariant

In [ ]:
deck = DeckFactory.standard()


simulator = Simulator(
    generators=[
        InShuffle(),
        OutShuffle()
    ],
    action=ShuffleAction()
)


final_deck, history = simulator.run(
    deck,
    steps=100
)

In [ ]:
combined_shuffle = CompositeShuffle(
    *history
)

In [ ]:
final_deck2 = ShuffleAction().apply(
    deck,
    combined_shuffle
)

## Some notes on Attempt 1:

The design above is mathematically clean, but I would refine it. The current version is a good **first decomposition**, but it has some unnecessary coupling from a software design perspective.

The main issue:

> We are representing a shuffle both as a function (P\to P) and as an object, but we have not fully separated **algebra**, **interpretation**, and **execution**.

A stronger design is closer to the categorical structure:

$$
\boxed{
Command\ (syntax)
\rightarrow
Permutation\ (semantics)
\rightarrow
Action\ (execution)
\rightarrow
Observation\ (invariants)
}
$$

---

# 1. The biggest change: introduce `Permutation`

Currently:

```python
class Shuffle:
    def __call__(self, position):
        ...
```

This mixes two things:

* generating a permutation
* being a permutation

A better model:

$$
Permutation = Bij(P,P)
$$

is the mathematical object.

---

## Permutation class

```python
from typing import Callable


class Permutation:

    def __init__(
        self,
        mapping: Callable[[int], int]
    ):
        self.mapping = mapping


    def __call__(self, x:int)->int:
        return self.mapping(x)


    def compose(
        self,
        other:"Permutation"
    ) -> "Permutation":

        return Permutation(
            lambda x:
                self(other(x))
        )
```

Now composition belongs here:

$$
S_{52}\times S_{52}\rightarrow S_{52}
$$

not in `Deck`.

---

# 2. Shuffles become generators of permutations

Instead of:

```python
OutShuffle()
```

being a permutation, make it a constructor:

```python
class ShuffleGenerator:

    def outshuffle(self):

        return Permutation(
            lambda i:
                2*i
                if i < 26
                else 2*(i-26)+1
        )


    def inshuffle(self):

        return Permutation(
            lambda i:
                2*i+1
                if i < 26
                else 2*(i-26)
        )
```

Now:

```python
out = generator.outshuffle()
```

means:

$$
out\in S_{52}
$$

---

# 3. Composite becomes trivial

Because permutations already form a monoid:

```python
sigma = (
    out
    .compose(in)
    .compose(out)
)
```

Mathematically:

$$
\sigma=out\circ in\circ out
$$

No special `CompositeShuffle` class needed.

This is an important simplification.

The category already gives you composition.

---

# 4. Deck should only represent state

Your deck:

$$
d:P\rightarrow C
$$

should not know about permutations.

```python
class Deck(BaseModel):

    cards: tuple[Card,...]

```

No:

```python
deck.shuffle()
```

because then the domain object starts owning external transformations.

---

# 5. Introduce an Action object

Now:

$$
S_{52}\times Deck\rightarrow Deck
$$

gets its own place.

```python
class DeckAction:


    def apply(
        self,
        permutation:Permutation,
        deck:Deck
    ):

        result=[None]*52


        for position, card in deck:

            new_position = permutation(position)

            result[new_position]=card


        return Deck(
            cards=tuple(result)
        )
```

Now the ownership is mathematically correct.

---

# 6. Command becomes useful here

Previously Command was slightly redundant.

Now it has a real role.

A command is not a shuffle.

A command is:

$$
\text{operation + metadata}
$$

Example:

```python
class ShuffleCommand:


    def __init__(
        self,
        permutation,
        name
    ):
        self.permutation = permutation
        self.name=name


    def execute(
        self,
        action,
        deck
    ):

        return action.apply(
            self.permutation,
            deck
        )
```

Now you can have:

```python
commands=[
    ShuffleCommand(out,"out"),
    ShuffleCommand(in_,"in")
]
```

This is your free monoid:

$$
G^*
$$

---

# 7. Simulator becomes a command interpreter

Now the simulator does:

$$
G^*
\rightarrow
Deck
$$

```python
class Simulator:

    def __init__(self, action):
        self.action=action


    def run(
        self,
        deck,
        commands
    ):

        current=deck


        for command in commands:

            current=command.execute(
                self.action,
                current
            )


        return current
```

---

# 8. Add invariants as separate observers

Do not attach:

```python
deck.sign()
```

because sign is not a property of a deck.

It is a property of a permutation.

The sign map is:

$$
sgn:S_{52}\rightarrow C_2
$$

Create:

```python
class PermutationInvariant:

    def compute(self, permutation):
        pass



class SignInvariant:

    def compute(self, permutation):

        cycles = cycle_count(permutation)

        return (-1)**(52-cycles)
```

Usage:

```python
sign = SignInvariant()

print(
    sign.compute(out)
)
```

---

# Final architecture

```mermaid
flowchart TD
    A["Shuffle Generators<br/>in/out"]
    B["Permutation<br/>S52"]
    C["Command Sequence<br/>G*"]
    D["DeckAction<br/>S52 x Deck -> Deck"]
    E["Deck"]
    F["Observers<br/>sign/signature"]

    A --> B
    A --> C
    C --> D
    B --> D
    D --> E
    B --> F
```

---

# Why this is better

| Component     | Owns                     | Does not own             |
| ------------- | ------------------------ | ------------------------ |
| `Permutation` | composition              | cards                    |
| `Deck`        | card state/invariant     | shuffling                |
| `Action`      | applying transformations | defining transformations |
| `Command`     | history/program          | execution mechanics      |
| `Simulator`   | orchestration            | math                     |
| `Invariant`   | measurement              | mutation                 |

---

The categorical correspondence is now almost exact:

$$
\begin{aligned}
G &\rightarrow \text{generators}\\
G^* &\rightarrow \text{commands}\\
\rho:G^*\rightarrow S_{52}&\rightarrow\text{interpretation}\\
S_{52}\times Deck\rightarrow Deck&\rightarrow\text{action}\\
S_{52}\rightarrow C_2&\rightarrow\text{invariant}
\end{aligned}
$$

This is the design I would use if the goal is not just "make a card simulator" but "make a system whose software structure mirrors the algebra".


## Attempt 2

The refined design has these layers:

$$\text{Generator} \to \text{Permutation} \to \text{Command} \to \text{Action} \to \text{Deck} \to \text{Invariant}$$
	​


The important ownership rules:

Permutation owns composition.
Deck owns only card state.
DeckAction owns the action 
$$\text{S}_{52} \times \text{Deck} \to \text{Deck}$$
Command owns history/sequence.
Simulator owns orchestration.
Invariant observes.

Here is the complete code.

In [ ]:
from enum import Enum
from pydantic import BaseModel, ConfigDict, model_validator
from typing import Callable
import random


class Suit(str, Enum):
    HEARTS = "♥"
    SPADES = "♠"
    CLUBS = "♣"
    DIAMONDS = "♦"


class Rank(str, Enum):
    A = "A"
    K = "K"
    Q = "Q"
    J = "J"
    TEN = "10"
    NINE = "9"
    EIGHT = "8"
    SEVEN = "7"
    SIX = "6"
    FIVE = "5"
    FOUR = "4"
    THREE = "3"
    TWO = "2"


class Card(BaseModel):
    model_config = ConfigDict(frozen=True)

    suit: Suit
    rank: Rank


STANDARD_CARDS = tuple(
    Card(suit=s, rank=r)
    for s in Suit
    for r in Rank
)

## Deck

Mathematically:

$$\mathtt{Deck} \simeq \mathtt{Bij}(\mathtt{P}, \mathtt{C})$$